In [ ]:
import pandas as pd
import numpy as np

# 1. LADEN

df = pd.read_csv('/Users/isabella/Downloads/archive (1)/btc_1h_data_2018_to_2025.csv')

# Timestamp parsen + als Index setzen
df["Open time"] = pd.to_datetime(df["Open time"])
df = df.set_index("Open time").sort_index()

print(f"Shape: {df.shape}")
print(f"Zeitraum: {df.index.min()} → {df.index.max()}")
print(f"\nDatentypen:\n{df.dtypes}")


Shape: (71639, 11)
Zeitraum: 2018-01-01 00:00:00 → 2026-03-09 21:00:00

Datentypen:
Open                            float64
High                            float64
Low                             float64
Close                           float64
Volume                          float64
Close time                       object
Quote asset volume              float64
Number of trades                  int64
Taker buy base asset volume     float64
Taker buy quote asset volume    float64
Ignore                            int64
dtype: object


In [3]:
# 2. CREATE FULL HOURLY INDEX
full_index = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="h"
)

expected_rows = len(full_index)
actual_rows   = len(df)

print(f"Expected rows (no gaps): {expected_rows}")
print(f"Actual rows:             {actual_rows}")
print(f"Missing hours:           {expected_rows - actual_rows}")

Expected rows (no gaps): 71758
Actual rows:             71639
Missing hours:           119


In [ ]:
# # 3. REINDEX + IDENTIFY GAPS
# df_full = df.reindex(full_index)

# missing_mask = df_full["Close"].isna()

# # Find consecutive gap groups
# gaps = []
# in_gap = False
# gap_start = None
# gap_len = 0

# for ts, is_missing in missing_mask.items():
#     if is_missing and not in_gap:
#         in_gap = True
#         gap_start = ts
#         gap_len = 1
#     elif is_missing and in_gap:
#         gap_len += 1
#     elif not is_missing and in_gap:
#         gaps.append({"start": gap_start, "end": ts, "duration_hours": gap_len})
#         in_gap = False
#         gap_len = 0

# # Last gap if data ends with missing values
# if in_gap:
#     gaps.append({"start": gap_start, "end": missing_mask.index[-1], "duration_hours": gap_len})

# gaps_df = pd.DataFrame(gaps).sort_values("duration_hours", ascending=False)

# print(f"Number of gaps: {len(gaps_df)}")
# print(f"Longest gap: {gaps_df['duration_hours'].max()} hours")
# print(f"Gaps > 3 hours: {len(gaps_df[gaps_df['duration_hours'] > 3])}")
# print(f"\nTop 10 longest gaps:\n{gaps_df.head(10).to_string(index=False)}")

ValueError: cannot reindex on an axis with duplicate labels

In [5]:
# ── 3a. INVESTIGATE DUPLICATE TIMESTAMPS ─────────────────────────────────────
dupes_mask = df.index.duplicated(keep=False)  # mark ALL occurrences
dupes = df[dupes_mask].sort_index()

print(f"Total rows involved in duplicates: {len(dupes)}")
print(f"Number of duplicate timestamps: {df.index.duplicated().sum()}")
print(f"\nDuplicate rows:\n{dupes[['Open', 'High', 'Low', 'Close', 'Volume']].to_string()}")

Total rows involved in duplicates: 4
Number of duplicate timestamps: 2

Duplicate rows:
                          Open       High        Low      Close      Volume
Open time                                                                  
2018-07-07 00:00:00    6609.79    6613.99    6581.05    6596.93  766.671591
2018-07-07 00:00:00  107469.22  107813.04  107440.63  107692.01  228.631950
2018-07-07 01:00:00    6596.93    6600.00    6561.01    6566.01  887.436249
2018-07-07 01:00:00  107692.01  107941.99  107689.91  107911.20  255.841360


In [ ]:
#3b. REMOVE DUPLICATE TIMESTAMPS
# Keep first occurrence — the ~6600 USD rows are correct for July 2018
# The ~107000 USD rows are clearly misplaced (impossible price for 2018)
df = df[~df.index.duplicated(keep="first")]

print(f"Shape after removing duplicates: {df.shape}")
print(f"Remaining duplicates: {df.index.duplicated().sum()}")

Shape after removing duplicates: (71637, 11)
Remaining duplicates: 0


In [7]:
#3c. REINDEX + IDENTIFY GAPS
df_full = df.reindex(full_index)

missing_mask = df_full["Close"].isna()

# Find consecutive gap groups
gaps = []
in_gap = False
gap_start = None
gap_len = 0

for ts, is_missing in missing_mask.items():
    if is_missing and not in_gap:
        in_gap = True
        gap_start = ts
        gap_len = 1
    elif is_missing and in_gap:
        gap_len += 1
    elif not is_missing and in_gap:
        gaps.append({"start": gap_start, "end": ts, "duration_hours": gap_len})
        in_gap = False
        gap_len = 0

# Last gap if data ends with missing values
if in_gap:
    gaps.append({"start": gap_start, "end": missing_mask.index[-1], "duration_hours": gap_len})

gaps_df = pd.DataFrame(gaps).sort_values("duration_hours", ascending=False)

print(f"Number of gaps: {len(gaps_df)}")
print(f"Longest gap: {gaps_df['duration_hours'].max()} hours")
print(f"Gaps > 3 hours: {len(gaps_df[gaps_df['duration_hours'] > 3])}")
print(f"\nTop 10 longest gaps:\n{gaps_df.head(10).to_string(index=False)}")

Number of gaps: 27
Longest gap: 33 hours
Gaps > 3 hours: 10

Top 10 longest gaps:
              start                 end  duration_hours
2018-02-08 01:00:00 2018-02-09 10:00:00              33
2018-06-26 02:00:00 2018-06-26 12:00:00              10
2019-05-15 03:00:00 2019-05-15 13:00:00              10
2019-08-15 02:00:00 2019-08-15 10:00:00               8
2018-07-04 01:00:00 2018-07-04 08:00:00               7
2018-11-14 02:00:00 2018-11-14 09:00:00               7
2019-03-12 02:00:00 2019-03-12 08:00:00               6
2020-02-19 12:00:00 2020-02-19 17:00:00               5
2020-12-21 14:00:00 2020-12-21 18:00:00               4
2021-08-13 02:00:00 2021-08-13 06:00:00               4


In [ ]:
#4. OHLC LOGIC CHECK
ohlc_violations = df[
    (df["High"] < df["Low"]) |
    (df["High"] < df["Open"]) |
    (df["High"] < df["Close"]) |
    (df["Low"]  > df["Open"]) |
    (df["Low"]  > df["Close"])
]
print(f"OHLC logic violations: {len(ohlc_violations)}")


OHLC logic violations: 0


In [9]:
# 5. EXTREME PRICE JUMPS
returns = df["Close"].pct_change()
extreme = returns[returns.abs() > 0.20]  # >20% in one hour

print(f"\nExtreme price jumps (>20% in 1h): {len(extreme)}")
if len(extreme) > 0:
    print(extreme.sort_values(key=abs, ascending=False).head(10))



Extreme price jumps (>20% in 1h): 0


In [10]:
# 6. NEGATIVE OR ZERO VALUES
price_cols = ["Open", "High", "Low", "Close", "Volume"]
print(f"\nNull values per column:\n{df[price_cols].isnull().sum()}")
print(f"\nZero or negative values per column:\n{(df[price_cols] <= 0).sum()}")


Null values per column:
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

Zero or negative values per column:
Open      0
High      0
Low       0
Close     0
Volume    3
dtype: int64


In [11]:
# 5. INVESTIGATE ZERO VOLUME ROWS
zero_volume = df[df["Volume"] == 0]

print(f"Zero volume rows:\n{zero_volume[['Open', 'High', 'Low', 'Close', 'Volume']].to_string()}")

Zero volume rows:
                         Open      High       Low     Close  Volume
Open time                                                          
2019-06-07 21:00:00   7930.85   7930.85   7930.85   7930.85     0.0
2021-02-11 03:00:00  44582.07  44582.07  44582.07  44582.07     0.0
2023-03-24 12:00:00  28080.00  28080.00  28080.00  28080.00     0.0


In [12]:
# 6. REPLACE ZERO VOLUME ROWS WITH NaN
# These rows have identical OHLC values and zero volume
# They are Binance placeholder candles — not real trades
df.loc[df["Volume"] == 0, ["Open", "High", "Low", "Close", "Volume"]] = np.nan

print(f"Rows with NaN Close after fix: {df['Close'].isna().sum()}")
print(f"Zero volume rows remaining: {(df['Volume'] == 0).sum()}")

Rows with NaN Close after fix: 3
Zero volume rows remaining: 0


In [13]:
# 7. FINAL DATASET SUMMARY
total_missing = df["Close"].isna().sum() + (expected_rows - len(df))

print("=" * 50)
print("DATASET QUALITY REPORT — BTC 1H 2018–2025")
print("=" * 50)
print(f"Total rows (after cleaning):  {len(df)}")
print(f"Time range:                   {df.index.min()} → {df.index.max()}")
print(f"Expected hours (full index):  {expected_rows}")
print(f"Missing hours (gaps):         {expected_rows - len(df)}")
print(f"Zero volume candles → NaN:    3")
print(f"Duplicate timestamps removed: 2")
print(f"OHLC violations:              0")
print(f"Extreme price jumps (>20%):   0")
print(f"Data completeness:            {len(df) / expected_rows:.4%}")
print("=" * 50)

DATASET QUALITY REPORT — BTC 1H 2018–2025
Total rows (after cleaning):  71637
Time range:                   2018-01-01 00:00:00 → 2026-03-09 21:00:00
Expected hours (full index):  71758
Missing hours (gaps):         121
Zero volume candles → NaN:    3
Duplicate timestamps removed: 2
OHLC violations:              0
Extreme price jumps (>20%):   0
Data completeness:            99.8314%
